# Combining and stacking dataframes
## Merging dataframes
### Basic Inner Merge (or inner join)

Only keeps rows that exist in BOTH datasets.

As you can see, property 4 (in `property` df) and property 6 (in `prices` df) disappear.

In [ ]:
import pandas as pd

properties = pd.DataFrame({
    "property_id": [1, 2, 3, 4, 5],
    "square_feet": [1800, 2200, 1200, 3000, 2500],
    "num_bedrooms": [3, 4, 2, 5, 4]
})

properties

In [ ]:
prices = pd.DataFrame({
    "property_id": [1, 2, 3, 5, 6],
    "price_thousand_usd": [250, 320, 180, 390, 410]
})

prices

In [ ]:
merged_inner = pd.merge(left=properties, right=prices, on="property_id", how='inner')
merged_inner

### Left join
- Keeps ALL rows from left dataset (properties)
- Missing matches become NaN
- Property 4 now has missing price

In [ ]:
merged_left = pd.merge(left=properties, right=prices, on="property_id", how="left")
merged_left

### Right join
- Keeps all rows from `prices`
- Property 6 appears with missing property info

In [ ]:
merged_right = pd.merge(left=properties, right=prices, on="property_id", how="right")
merged_right

### Outer Join (Full Join)
- Keeps everything
- Good for detecting mismatches
- Often used in data cleaning

In [ ]:
merged_outer = pd.merge(left=properties, right=prices, on="property_id", how="outer")
merged_outer

You can also set `indicator=True` when you do outer join. This way, the new dataframe will have a new column called `_merge`, which is a categorical variable that has three possible values: `both`, `left_only`, `right_only`.

In [ ]:
merged_outer = pd.merge(left=properties, right=prices,
                        on="property_id", how="outer", indicator=True)
merged_outer

### Merge on Different Column Names

- Column names do not need to match
- Use `left_on` and `right_on`

In [ ]:
prices_renamed = pd.DataFrame({
    "id": [1, 2, 3, 5, 6],
    "price_thousand_usd": [250, 320, 180, 390, 410]
})

pd.merge(left=properties, right=prices_renamed,
         left_on="property_id", right_on="id", how="inner")

### Merge on Multiple Keys

- Real datasets often require multiple keys
- Prevents incorrect duplication

In [ ]:
sales = pd.DataFrame({
    "property_id": [1, 1, 2, 3],
    "year": [2023, 2024, 2024, 2024],
    "price_thousand_usd": [240, 250, 320, 180]
})

tax_rates = pd.DataFrame({
    "property_id": [1, 2, 3],
    "year": [2024, 2024, 2024],
    "tax_rate": [0.012, 0.011, 0.013]
})

pd.merge(left=sales, right=tax_rates,
         on=["property_id", "year"], how="left")

## Joining dataframes
### Basic Index-Based Join

In [ ]:
# Dataset 1: Property Info
import pandas as pd

properties = pd.DataFrame({
    "property_id": [1, 2, 3, 4],
    "square_feet": [1800, 2200, 1200, 3000],
    "num_bedrooms": [3, 4, 2, 5]
}).set_index("property_id")

properties

In [ ]:
# Dataset 2: Price Info
prices = pd.DataFrame({
    "property_id": [1, 2, 3],
    "price_thousand_usd": [250, 320, 180]
}).set_index("property_id")

prices

In [ ]:
# Joins on index automatically. Equivalent to a left join. Property 4 gets NaN price.
properties.join(prices)

### Join Without Setting Index (Using `on=`)

The difference is, this time, `properties` does not have `property_id` as index
- `on=` specifies which column in left DataFrame matches the right index.
- Right DataFrame must have matching index.


In [ ]:
import pandas as pd

properties = pd.DataFrame({
    "property_id": [1, 2, 3, 4],
    "square_feet": [1800, 2200, 1200, 3000],
    "num_bedrooms": [3, 4, 2, 5]
})

prices = pd.DataFrame({
    "property_id": [1, 2, 3],
    "price_thousand_usd": [250, 320, 180]
}).set_index("property_id")


In [ ]:
properties.join(prices, on="property_id")

### [Optional] Join Multiple DataFrames at Once
- `.join()` can take a list of DataFrames.
- Very convenient for combining many datasets with same index.
- Cleaner than multiple merges.

In [ ]:
import pandas as pd

properties = pd.DataFrame({
    "property_id": [1, 2, 3, 4],
    "square_feet": [1800, 2200, 1200, 3000],
    "num_bedrooms": [3, 4, 2, 5]
}).set_index("property_id")

prices = pd.DataFrame({
    "property_id": [1, 2, 3],
    "price_thousand_usd": [250, 320, 180]
}).set_index("property_id")

tax = pd.DataFrame({
    "property_id": [1, 2, 3, 4],
    "tax_rate": [0.012, 0.011, 0.013, 0.014]
}).set_index("property_id")


In [ ]:
properties.join([prices, tax])

## Concatenate dataframes
### Row-wise Concatenation (Most Common)
Scenario: Two datasets from different years, same structure.


In [ ]:
import pandas as pd

# Dataset 1 (2023 Sales)
sales_2023 = pd.DataFrame({
    "property_id": [1, 2, 3],
    "price_thousand_usd": [240, 310, 175]
})

sales_2023

In [ ]:
# Dataset 2 (2024 Sales)
sales_2024 = pd.DataFrame({
    "property_id": [4, 5],
    "price_thousand_usd": [400, 285]
})

sales_2024

Concatenate Rows
- Default is `axis=0`, stack rows.
- Indices are preserved (may duplicate).
- Structures must match.

In [ ]:
pd.concat([sales_2023, sales_2024])

In [ ]:
# Clean Version (Reset Index). ignore_index=True avoids duplicate indices.
pd.concat([sales_2023, sales_2024], ignore_index=True)

### Column-wise Concatenation

Scenario: We computed features separately and want to combine them.

In [ ]:
import pandas as pd

df_a = pd.DataFrame({
    "square_feet": [1800, 2200, 1200]
})

df_b = pd.DataFrame({
    "num_bedrooms": [3, 4, 2]
})

Concatenate Columns
- `axis=1` means adding columns.
- Rows must align by index.
- Very common in feature engineering.

In [ ]:
pd.concat([df_a, df_b], axis=1)

### Mismatched Columns

In [ ]:
import pandas as pd

df1 = pd.DataFrame({
    "property_id": [1, 2],
    "price": [250, 320]
})

df1

In [ ]:
df2 = pd.DataFrame({
    "property_id": [3, 4],
    "square_feet": [1800, 2200]
})

df2


Concatenate the two dataframes that have different rows:
- Missing columns filled with NaN.
- Good for combining slightly different structures.
- Risky if columns accidentally mismatched.

In [ ]:
pd.concat([df1, df2], ignore_index=True)

You can also control Join Type:
- By default, concat keeps all columns (`join="outer"`).
    - `join = 'outer'`: keep all columns
    - `join = 'inner'`: keep shared columns only
- You can restrict to common columns


In [ ]:
pd.concat([df1, df2], join="inner", ignore_index=True)